In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from google_play_scraper import Sort, reviews
from tqdm import tqdm
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

from utils import classify_sentiment

In [2]:
app_ids = {
    'ChatGPT': 'com.openai.chatgpt',
    'Gemini': 'com.google.android.apps.bard',
    'Claude': 'com.anthropic.claude'
}

In [3]:
reviews_per_app = 100000
all_reviews = []

In [4]:
for app_name, app_id in app_ids.items():
    print(f"Scraping reviews for {app_name}...")

    app_results = []
    continuation_token = None

    with tqdm(total=reviews_per_app, position=0, leave=True) as pbar:
        while len(app_results) < reviews_per_app:
            new_result, continuation_token = reviews(
                app_id,
                continuation_token=continuation_token,
                lang='en',
                country='us',
                sort=Sort.NEWEST,
                count=199,  # API limit
            )

            if not new_result:
                print(f"\nWarning: Ran out of reviews for {app_name}. Found {len(app_results)}.")
                break

            app_results.extend(new_result)
            pbar.update(len(new_result))

    # Cap at exactly reviews_per_app in case the last batch overshot.
    app_results = app_results[:reviews_per_app]

    for review in app_results:
        review['appName'] = app_name

    all_reviews.extend(app_results)


Scraping reviews for ChatGPT...


  0%|          | 0/100000 [00:00<?, ?it/s]

100097it [06:41, 249.28it/s]                           


Scraping reviews for Gemini...


100097it [07:45, 214.92it/s]                           


Scraping reviews for Claude...


 34%|███▎      | 33677/100000 [01:39<03:15, 340.06it/s]

In [5]:
df_reviews = pd.DataFrame(all_reviews)
output_file = 'ai_apps_reviews_100000.csv'
df_reviews.to_csv(output_file, index=False)
print(f"Saved {len(df_reviews)} total reviews across {len(app_ids)} apps to '{output_file}'.")
df_reviews.head()

Saved 233677 total reviews across 3 apps to 'ai_apps_reviews_100000.csv'.


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,appName
0,2874309c-1211-4cc5-8fbd-0f4dc99bc902,parimala madheshi,https://play-lh.googleusercontent.com/a-/ALV-U...,plan upgrade pending after paying amount of 399,1,0,1.2026.097,2026-04-18 12:24:35,NaN,NaT,1.2026.097,ChatGPT
1,553c536a-0621-46c3-b779-745d506db4b2,Prateek Pandey,https://play-lh.googleusercontent.com/a-/ALV-U...,👌👌👌👌👌👌👌,5,0,1.2026.097,2026-04-18 12:24:14,NaN,NaT,1.2026.097,ChatGPT
2,c48ba7b3-782d-4468-a62f-d96cd3602748,sammy wanjala,https://play-lh.googleusercontent.com/a-/ALV-U...,helps me with my assignment and it's very cons...,5,0,1.2026.097,2026-04-18 12:23:58,NaN,NaT,1.2026.097,ChatGPT
3,7b68ad01-d722-4b89-8b26-598cd77778bd,YOGESH NIKURE,https://play-lh.googleusercontent.com/a/ACg8oc...,this is the best ai and i am glad to I have th...,5,0,1.2026.090,2026-04-18 12:23:56,NaN,NaT,1.2026.090,ChatGPT
4,0be4eb54-c741-478a-995f-a95d86726628,Ankush Sirsat,https://play-lh.googleusercontent.com/a-/ALV-U...,nice app,5,0,1.2026.097,2026-04-18 12:23:45,NaN,NaT,1.2026.097,ChatGPT


In [6]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

df_reviews['Sentiment'] = df_reviews['score'].apply(classify_sentiment)
print(f"Class counts (natural distribution): {df_reviews['Sentiment'].value_counts().to_dict()}")

Class counts (natural distribution): {'Positive': 194792, 'Negative': 28566, 'Neutral': 10319}


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jakobaune/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    return " ".join(w for w in words if w not in stop_words)

df_model = df_reviews.dropna(subset=['appName', 'content', 'Sentiment']).copy()
df_model['content'] = df_model['content'].astype(str).str.strip()
df_model = df_model[df_model['content'] != ""]

before_dedup = len(df_model)
df_model = df_model.drop_duplicates(subset=['appName', 'content'], keep='first').reset_index(drop=True)
removed_duplicates = before_dedup - len(df_model)

print(f"Rows after filtering: {before_dedup}")
print(f"Removed duplicates: {removed_duplicates}")
print(f"Rows after dedup: {len(df_model)}")

print("Cleaning text...")
df_model['clean_content'] = df_model['content'].apply(clean_text)

# Drop rows where clean_content is empty — empty strings become NaN
before_empty = len(df_model)
df_model = df_model[df_model['clean_content'].str.strip() != ''].reset_index(drop=True)
print(f"Removed {before_empty - len(df_model)} rows with empty clean_content")

# Split FIRST, balance LATER (each downstream notebook decides whether/how to balance its own training partition).
# Stratified so both partitions preserve the natural class distribution.
train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42,
    stratify=df_model['Sentiment'],
)

print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")
print("Train counts:", train_df['Sentiment'].value_counts().to_dict())
print("Test counts:", test_df['Sentiment'].value_counts().to_dict())

train_df.to_csv('train_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)
print("Saved train_data.csv and test_data.csv.")

Rows after filtering: 233676
Removed duplicates: 109896
Rows after dedup: 123780
Cleaning text...
Removed 1490 rows with empty clean_content

Train: 97832 | Test: 24458
Train counts: {'Positive': 75619, 'Negative': 17431, 'Neutral': 4782}
Test counts: {'Positive': 18905, 'Negative': 4357, 'Neutral': 1196}
Saved train_data.csv and test_data.csv.
